### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Step 2: Open the project directory

Replace `Your_Dir` with your own path.

In [2]:
cd /content/drive/MyDrive/ECE_C247A_Final_Project/

/content/drive/MyDrive/ECE_C247A_Final_Project


### Step 3: Install required packages

After installing them, Colab will require you to restart the session.

In [3]:
!pip install -r requirements.txt

     - 553.6 kB 582.4 kB/s 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This coul

### Step 4: Start your experiments!

- Remember to download and copy the dataset to this directory: `Your_Dir/emg2qwerty/data`.
- You may now start your experiments with any scripts! Below are examples of single-user training and testing (greedy decoding).
- **There are two ways to track the logs:**
  - 1. Keep `--multirun`, and the logs will not be printed here, but they will be saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/submitit_logs/`.
  - 2. Comment out `--multirun` and the logs will be printed in this notebook, but they will not be saved.

#### Training

- The checkpoints are saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/checkpoints/`.

In [ ]:
import subprocess
from pathlib import Path
import pandas as pd
import re

repo_dir = Path("/content/drive/MyDrive/ECE_C247A_Final_Project")

channel_sets = {
    16: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
    12: [0, 1, 2, 4, 5, 6, 8, 9, 10, 12, 13, 14],
    8:  [0, 2, 4, 6, 8, 10, 12, 14],
    4:  [0, 4, 8, 12],
}

results = []

def parse_float_from_line(line: str):
    matches = re.findall(r"[-+]?\d*\.\d+|\d+", line)
    return float(matches[-1]) if matches else None

for n_channels, channels in channel_sets.items():
    in_features = n_channels * 33

    print(f"\n{'='*60}")
    print(f"Running experiment with {n_channels} channels")
    print(f"Selected channels: {channels}")
    print(f"in_features: {in_features}")
    print(f"{'='*60}\n")

    cmd = [
        "python", "-u", "-m", "emg2qwerty.train",
        "user=single_user",
        "trainer.accelerator=gpu",
        "trainer.devices=1",
        "trainer.max_epochs=50",
        "num_workers=2",
        "module.lstm_hidden_size=256",
        "module.lstm_num_layers=2",
        "module.lstm_dropout=0.2",
        "optimizer.lr=0.0005",
        f"module.electrode_channels={n_channels}",
        f"module.in_features={in_features}",
        f"module.selected_channels={channels}",
    ]

    best_val_cer = None
    final_val_cer = None
    final_test_cer = None

    process = subprocess.Popen(
        cmd,
        cwd=repo_dir,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        print(line, end="")

        if "'val/CER' reached" in line:
            try:
                cer_str = line.split("reached")[1].split("(")[0].strip()
                best_val_cer = float(cer_str)
            except:
                pass

        if "│          val/CER" in line:
            val = parse_float_from_line(line)
            if val is not None:
                final_val_cer = val

        if "│         test/CER" in line:
            val = parse_float_from_line(line)
            if val is not None:
                final_test_cer = val

    return_code = process.wait()

    if return_code != 0:
        print(f"\nRun failed for {n_channels} channels\n")
        results.append({
            "channels": n_channels,
            "selected_channels": channels,
            "in_features": in_features,
            "best_val_CER": None,
            "final_val_CER": None,
            "final_test_CER": None,
        })
    else:
        results.append({
            "channels": n_channels,
            "selected_channels": channels,
            "in_features": in_features,
            "best_val_CER": best_val_cer,
            "final_val_CER": final_val_cer,
            "final_test_CER": final_test_cer,
        })

    results_df = pd.DataFrame(results).sort_values("channels", ascending=False)
    display(results_df)

results_df = pd.DataFrame(results).sort_values("channels", ascending=False)
results_df.to_csv(repo_dir / "channel_ablation_results.csv", index=False)

print("\nSaved summary to:")
print(repo_dir / "channel_ablation_results.csv")
display(results_df)


Running experiment with 16 channels
Selected channels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
in_features: 528

[2026-03-12 09:23:06,817][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-91

,channels,selected_channels,in_features,best_val_CER,final_val_CER,final_test_CER
0,16,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",528,17.34604,17.346035,19.250757



Running experiment with 12 channels
Selected channels: [0, 1, 2, 4, 5, 6, 8, 9, 10, 12, 13, 14]
in_features: 396

[2026-03-12 10:05:17,645][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5

,channels,selected_channels,in_features,best_val_CER,final_val_CER,final_test_CER
0,16,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",528,17.34604,17.346035,19.250757
1,12,"[0, 1, 2, 4, 5, 6, 8, 9, 10, 12, 13, 14]",396,19.60567,19.605671,21.004765



Running experiment with 8 channels
Selected channels: [0, 2, 4, 6, 8, 10, 12, 14]
in_features: 264

[2026-03-12 10:47:21,684][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89

,channels,selected_channels,in_features,best_val_CER,final_val_CER,final_test_CER
0,16,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",528,17.34604,17.346035,19.250757
1,12,"[0, 1, 2, 4, 5, 6, 8, 9, 10, 12, 13, 14]",396,19.60567,19.605671,21.004765
2,8,"[0, 2, 4, 6, 8, 10, 12, 14]",264,20.04874,20.048737,22.867043



Running experiment with 4 channels
Selected channels: [0, 4, 8, 12]
in_features: 132

[2026-03-12 11:29:26,375][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    ses

,channels,selected_channels,in_features,best_val_CER,final_val_CER,final_test_CER
0,16,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",528,17.34604,17.346035,19.250757
1,12,"[0, 1, 2, 4, 5, 6, 8, 9, 10, 12, 13, 14]",396,19.60567,19.605671,21.004765
2,8,"[0, 2, 4, 6, 8, 10, 12, 14]",264,20.04874,20.048737,22.867043
3,4,"[0, 4, 8, 12]",132,26.56181,26.561808,27.479429



Saved summary to:
/content/drive/MyDrive/ECE_C247A_Final_Project/channel_ablation_results.csv


,channels,selected_channels,in_features,best_val_CER,final_val_CER,final_test_CER
0,16,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",528,17.34604,17.346035,19.250757
1,12,"[0, 1, 2, 4, 5, 6, 8, 9, 10, 12, 13, 14]",396,19.60567,19.605671,21.004765
2,8,"[0, 2, 4, 6, 8, 10, 12, 14]",264,20.04874,20.048737,22.867043
3,4,"[0, 4, 8, 12]",132,26.56181,26.561808,27.479429


In [ ]:
from google.colab import runtime
runtime.unassign()